In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class Encoder(nn.Module):
  def __init__(self,vocab_size,hidden_sizes,dropout_rates,embed_dim):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embed_dim)
    self.lstms = nn.ModuleList()
    self.dropouts=nn.ModuleList()
    current_input_size=embed_dim
    for hidden_size,dropout_rate in zip(hidden_sizes,dropout_rates):
      self.lstms.append(nn.LSTM(current_input_size,hidden_size,batch_first=True))
      self.dropouts.append(nn.Dropout(dropout_rate))
      current_input_size=hidden_size
  def forward(self,x):
    x=self.embedding(x)
    all_hidden_states=[]
    all_cell_states=[]
    for lstm,dropout in zip(self.lstms,self.dropouts):
      x=dropout(x)
      x,(h,c)=lstm(x)
      all_hidden_states.append(h)
      all_cell_states.append(c)
    final_hidden_state=torch.cat(all_hidden_states,dim=0)
    final_cell_state=torch.cat(all_cell_states,dim=0)
    return final_hidden_state,final_cell_state


In [ ]:
class Attention(nn.Module):
  def __init__(self,encoder_hidden_dim,decoder_hidden_dim):
    super().__init__()
    self.Encoder_W=nn.Linear(encoder_hidden_dim,decoder_hidden_dim)
    self.Decoder_W=nn.Linear(decoder_hidden_dim,decoder_hidden_dim)
    self.proj=nn.Linear(decoder_hidden_dim,1,bias=False)

  def forward(self,top_encoder_outputs,top_decoder_output):
    proj_decoder=self.Decoder_W(top_decoder_output.unsqueeze(1))
    proj_encoder=self.Encoder_W(top_encoder_outputs)

    attn_scores=self.proj(torch.tanh(proj_decoder+proj_encoder)).squeeze(2)       ### Make it [batch_size,seq_len]
    attn_weights=torch.nn.functional.softmax(attn_scores,dim=1)

    context=torch.bmm(attn_weights.unsqueeze(1),top_encoder_outputs)
    return context,attn_weights

In [ ]:
class Decoder(nn.Module):
  def __init__(self,vocab_size,decoder_hidden_sizes,encoder_hidden_sizes,dropout_rates,embed_dim):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embed_dim)
    self.lstms=nn.ModuleList()
    self.dropouts=nn.ModuleList()
    self.attention=Attention(encoder_hidden_sizes[-1],decoder_hidden_sizes[-1])
    current_input_size=embed_dim+encoder_hidden_sizes[-1]
    for hidden_size,dropout_rate in zip(decoder_hidden_sizes,dropout_rates):
      self.lsmts.append(nn.LSTM(current_input_size,hidden_size,batch_first=True))
      self.dropouts.append(nn.Dropout(dropout_rate))
      current_input_size=hidden_size
    self.fc=nn.Linear(current_input_size+encoder_hidden_sizes,vocab_size)
  def forward(self,x,encoder_hidden,encoder_cell):
    x=self.embedding(x)
    all_hidden_states=[]
    all_cell_states=[]
    for i,(lstm,dropout) in enumerate(zip(self.lstms,self.dropouts)):
      x=dropout(x)
      x,(h,c)=lstm(x,(encoder_hidden[i].unsqueeze(0),encoder_cell[i].unsqueeze(0)))
      all_hidden_states.append(h)
      all_cell_states.append(c)
    predictions=self.fc(x)
    final_hidden=torch.cat(all_hidden_states,dim=0)
    final_cell=torch.cat(all_cell_states,dim=0)
    return predictions,final_hidden,final_cell

In [ ]:
class seq2seq_training(nn.Module):
  def __init__(self,encoder,decoder,vocab_size):
    super().__init__()
    self.encoder=encoder
    self.decoder=decoder
    self.vocab_size=vocab_size
  def forward(self,source,target):
    batch_size=source.shape[0]
    target_len=target.shape[1]
    outputs=torch.zeros(batch_size,target_len,self.vocab_size).to(device)
    hidden,cell=encoder(source)
    decoder_input=target[:,0].unsqueeze(0)
    for i in range(1,target_len):
      output,hidden,cell=decoder(decoder_input,hidden,cell)
      outputs[:,i,:]=output.squeeze(1)
      decoder_input=target[:,i].unsqueeze(0)
    return outputs

In [ ]:
class seq2seq_inference(nn.Module):
  def __init__(self,encoder,decoder,vocab_size):
    super().__init__()
    self.encoder=encoder
    self.decoder=decoder
    self.vocab_size=vocab_size
  def forward(self,source,target):
    batch_size=source.shape[0]
    target_len=target.shape[1]
    outputs=torch.zeros(batch_size,target_len,self.vocab_size).to(device)
    hidden,cell=encoder(source)
    decoder_input=target[:,0].unsqueeze(0)
    for i in range(1,target_len):
      output,hidden,cell=decoder(decoder_input,hidden,cell)
      outputs[:,i,:]=output.squeeze(1)
      top_prediction=output.argmax(2)
      decoder_input=top_prediction
    return outputs

In [ ]:
enc=Encoder(vocab_size,hidden_sizes,dropout_rates,embed_dim)
dec=Decoder(vocab_size,hidden_sizes,dropout_rates,embed_dim)
model=seq2seq_training(enc,dec,vocab_size)
model.to(device)

In [ ]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(learning_rate=0.01,params=model.parameters())

In [ ]:
model.train()
for i in range(epochs):
  total_loss=0
  for batch_x,batch_y in train_loader:
    batch_x=batch_x.to(device)
    batch_y=batch_y.to(device)
    predictions=model(batch_x,batch_y)
    loss=criterion(predictions[:,1:].reshape(-1,predictions.shape[-1]),batch_y[:,1:].reshape(-1))
    loss.backward()
    optimizer.zero_grad()
    optimizer.step()
    total_loss+=loss.item()
  print(f"epoch:{i+1},loss:{total_loss/len(train_loader)}")
